# Autonomous Crypto Trading Lab v3
Fresh launcher with cache-safe imports, eight research slots, and independent confirmation.


In [ ]:
import pathlib, shutil, subprocess, sys, os, importlib
REPO_URL='https://github.com/betaanoiar1-gif/autonomous-crypto-trading-lab.git'
REPO_DIR='/content/autonomous_crypto_trading_lab_v3'
print('=== Autonomous Crypto Trading Lab v3 ===')
if pathlib.Path(REPO_DIR).exists(): shutil.rmtree(REPO_DIR)
print('Cloning fresh project...')
subprocess.run(['git','clone','--depth','1',REPO_URL,REPO_DIR],check=True)
rev=subprocess.check_output(['git','-C',REPO_DIR,'rev-parse','--short','HEAD'],text=True).strip()
print(f'Project revision: {rev}')
subprocess.run([sys.executable,'-m','pip','install','-q','-r',f'{REPO_DIR}/requirements.txt'],check=True)
if REPO_DIR in sys.path: sys.path.remove(REPO_DIR)
sys.path.insert(0,REPO_DIR)
for name in list(sys.modules):
    if name == 'lab' or name.startswith('lab.'):
        del sys.modules[name]
importlib.invalidate_caches()
os.environ['AGENT_PROVIDER']='local'
print(f'Import root: {REPO_DIR}')
from lab.local_agent import LocalAgent
agent=LocalAgent()
health=agent.healthcheck()
print(f"Local agent: {'OK' if health['ok'] else 'FAILED'} | model={health['model']} | device={health.get('device','unknown')}")
if not health['ok']: raise RuntimeError('Local research agent failed to load.')
from lab.research.run import DIVERSITY_SLOTS, run
target=min(8,len(DIVERSITY_SLOTS))
print(f'Research slots requested: {target}')
print('Launching autonomous research...')
result=run(max_hypotheses=target,agent=agent)
print(f"Research run: {result['run_id']}")
print(f"Hypotheses generated: {result['hypothesis_count']}")
print(f"Statuses: {[r.get('status','UNKNOWN') for r in result['records']]}")
print('Results saved under experiments/')


In [ ]:
from lab.data.ccxt_adapter import CCXTMarketData
from lab.research.confirmation import confirm_on_independent_market, confirmation_passed
from lab.config import load_settings
settings=load_settings()
print('=== Independent confirmation ===')
adapter=CCXTMarketData(exchange_id='binance')
seen=False
for record in result['records']:
    if record.get('status') != 'VALIDATION_CANDIDATE':
        continue
    seen=True
    symbol=record['symbol']
    timeframe=record['timeframe']
    alternate='ETH/USDT' if symbol=='BTC/USDT' else 'BTC/USDT'
    try:
        df=adapter.fetch_ohlcv_history(alternate,timeframe,target_rows={'15m':10000,'1h':5000,'4h':3000}[timeframe],market_type=record.get('market_type','spot'))
        split=max(1,int(len(df)*0.70))
        confirm_df=df.iloc[split:].copy()
        h=record['hypothesis']
        params=record.get('out_of_sample',{}).get('selected_parameters',h.get('executable_parameters',{}))
        directions=[d if isinstance(d,str) else str(d) for d in h.get('directions',['long'])]
        metrics=confirm_on_independent_market(confirm_df,h['executable_family'],params,directions,settings.capital.initial_usd,settings.execution.commission_bps,settings.execution.slippage_bps,market_type=record.get('market_type','spot'),leverage=1.0)
        ok=confirmation_passed(metrics)
        print(f"Confirmation: {h['title']} | {alternate} | {timeframe} | PASS={ok} | return={metrics['total_return']:.2%} | PF={metrics['profit_factor']:.2f} | DD={metrics['max_drawdown']:.2%} | trades={metrics['trade_count']}")
    except Exception as exc:
        print(f"Confirmation: {h.get('title','candidate')} | {alternate} | FAILED TO TEST | {exc}")
if not seen: print('No validation candidates require independent confirmation.')
print('=== Launch complete ===')
